In [ ]:
# import zipfile
# import os
# import requests

# # Caminho do arquivo ZIP
# zip_url = "https://caelum-online-public.s3.amazonaws.com/challenge-spark/semana-1.zip"
# zip_path = "/home/luizh/Python/1Challenge/Data Science/Alura-Challenge-DS/Semanas_zip/semana-1.zip"
# extract_path = "/home/luizh/Python/1Challenge/Data Science/Alura-Challenge-DS/Dataset"

# # Baixar o arquivo ZIP
# response = requests.get(zip_url)
# with open(zip_path, "wb") as f:
#     f.write(response.content)

# # Extrair o conteúdo
# with zipfile.ZipFile(zip_path, "r") as zip_ref:
#     zip_ref.extractall(extract_path)

# # Listar os arquivos extraídos
# os.listdir(extract_path)

['dataset_bruto.json']

In [ ]:
from pyspark.sql import SparkSession

# Inicializar o Spark
spark = SparkSession.builder.appName("ProcessamentoJSON").getOrCreate()

# Caminho do arquivo JSON
json_path = "/home/luizh/Python/1Challenge/Data Science/Alura-Challenge-DS/Dataset/dataset_bruto.json"

# Carregar o arquivo JSON em um DataFrame
df = spark.read.json(json_path)

# Exibir as primeiras linhas
df.show(10)

+--------------------+--------------------+--------------------+
|             anuncio|             imagens|             usuario|
+--------------------+--------------------+--------------------+
|{0, [], [16], [0]...|[{39d6282a-71f3-4...|{9d44563d-3405-4e...|
|{0, [], [14], [0]...|[{23d2b3ab-45b0-4...|{36245be7-70fe-40...|
|{0, [1026], [1026...|[{1da65baa-368b-4...|{9dc415d8-1397-4d...|
|{0, [120], [120],...|[{79b542c6-49b4-4...|{9911a2df-f299-4a...|
|{0, [3], [3], [0]...|[{e2bc497b-6510-4...|{240a7aab-12e5-40...|
|{0, [20], [15], [...|[{2de09d46-dc0d-4...|{3c7057f5-0923-42...|
|{3, [43], [43], [...|[{147a80d9-cd40-4...|{5a9736b5-aaa0-4a...|
|{2, [42], [42], [...|[{35740004-063d-4...|{ec48d96a-137c-49...|
|{0, [], [12], [0]...|[{6d3d2aec-c96f-4...|{dad7db63-e19c-44...|
|{1, [41], [41], [...|[{3d404069-418e-4...|{a845f35f-3ab3-46...|
+--------------------+--------------------+--------------------+
only showing top 10 rows



# Para nossa análise e tratamentos dos dados, a equipe solicitou que apenas a coluna "anuncio" será utilizado. Logo faremos a extração dela e transformaremos em um novo DataFrame

In [7]:
tabela = df.select('anuncio')
arrays_df = tabela.collect()
arrays_df[0]

Row(anuncio=Row(andar=0, area_total=[], area_util=['16'], banheiros=[0], caracteristicas=[], endereco=Row(bairro='Centro', cep='20061003', cidade='Rio de Janeiro', estado='Rio de Janeiro', latitude=-22.906082, longitude=-43.18671, pais='BR', rua='Rua Buenos Aires', zona='Zona Central'), id='47d553e0-79f2-4a46-9390-5a3c962740c2', quartos=[0], suites=[0], tipo_anuncio='Usado', tipo_unidade='Outros', tipo_uso='Comercial', vaga=[1], valores=[Row(condominio='260', iptu='107', tipo='Venda', valor='10000')]))

In [8]:
elementos = [array['anuncio'] for array in arrays_df]

df_final = spark.createDataFrame(elementos)
df_final.show(5)

25/11/08 21:16:01 WARN TaskSetManager: Stage 3 contains a task of very large size (4526 KiB). The maximum recommended task size is 1000 KiB.


+-----+----------+---------+---------+--------------------+--------------------+--------------------+-------+------+------------+------------+-----------+----+--------------------+
|andar|area_total|area_util|banheiros|     caracteristicas|            endereco|                  id|quartos|suites|tipo_anuncio|tipo_unidade|   tipo_uso|vaga|             valores|
+-----+----------+---------+---------+--------------------+--------------------+--------------------+-------+------+------------+------------+-----------+----+--------------------+
|    0|        []|     [16]|      [0]|                  []|{Centro, 20061003...|47d553e0-79f2-4a4...|    [0]|   [0]|       Usado|      Outros|  Comercial| [1]|[{260, 107, Venda...|
|    0|        []|     [14]|      [0]|                  []|{Centro, 20051040...|b6ffbae1-17f6-487...|    [0]|    []|       Usado|      Outros|  Comercial| [0]|[{260, 107, Venda...|
|    0|    [1026]|   [1026]|      [0]|                  []|{Maria da Graça, ...|1fb030a5-9e3e-4

# O time de Data Science solicitou que fizéssemos alguns filtros nas colunas `tipo_uso`, `tipo_unidade` e `tipo_anuncio` da nossa base de dados:

- tipo_uso: **Residencial**;

- tipo_unidade: **Apartamento**;

- tipo_anuncio: **Usado**.

In [9]:
df_final = df_final.select('*') \
                   .where((df_final['tipo_uso'] == 'Residencial') \
                           & (df_final['tipo_unidade'] == 'Apartamento') \
                           & (df_final['tipo_anuncio'] == 'Usado'))
df_final.show(5)

+-----+----------+---------+---------+--------------------+--------------------+--------------------+-------+------+------------+------------+-----------+----+--------------------+
|andar|area_total|area_util|banheiros|     caracteristicas|            endereco|                  id|quartos|suites|tipo_anuncio|tipo_unidade|   tipo_uso|vaga|             valores|
+-----+----------+---------+---------+--------------------+--------------------+--------------------+-------+------+------------+------------+-----------+----+--------------------+
|    3|      [43]|     [43]|      [1]|[Academia, Churra...|{Paciência, 23585...|d2e3a3aa-09b5-45a...|    [2]|    []|       Usado| Apartamento|Residencial| [1]|[{245, NULL, Vend...|
|    2|      [42]|     [42]|      [1]|[Churrasqueira, P...|{Paciência, 23585...|085bab2c-87ad-452...|    [2]|    []|       Usado| Apartamento|Residencial| [1]|[{0, 0, Venda, 15...|
|    1|      [41]|     [41]|      [1]|[Portaria 24h, Co...|{Guaratiba, 23036...|18d22cbe-1b86-4

25/11/08 21:16:02 WARN TaskSetManager: Stage 4 contains a task of very large size (4526 KiB). The maximum recommended task size is 1000 KiB.


In [10]:
df_final.printSchema()

root
 |-- andar: long (nullable = true)
 |-- area_total: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- area_util: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- banheiros: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- caracteristicas: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- endereco: struct (nullable = true)
 |    |-- bairro: string (nullable = true)
 |    |-- cep: string (nullable = true)
 |    |-- cidade: string (nullable = true)
 |    |-- estado: string (nullable = true)
 |    |-- latitude: double (nullable = true)
 |    |-- longitude: double (nullable = true)
 |    |-- pais: string (nullable = true)
 |    |-- rua: string (nullable = true)
 |    |-- zona: string (nullable = true)
 |-- id: string (nullable = true)
 |-- quartos: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- suites: array (nullable = true)
 |    |-- element: long (c

# Como é percepitível, algumas colunas estão com estrutura de array e isso atrapalha se quisermos fazer um modelo de machine learning posteriormente. Nesse contexto, transformaremos os dados das colunas "quartos", "suites", "banheiros", "vaga", "area_total" e "area_util" de listas para inteiros.

In [11]:
from pyspark.sql.types import IntegerType, StringType

In [12]:
df_final = df_final.withColumn("quartos",df_final.quartos[0].cast(IntegerType()))
df_final = df_final.withColumn("suites",df_final.suites[0].cast(IntegerType()))
df_final = df_final.withColumn("banheiros",df_final.banheiros[0].cast(IntegerType()))
df_final = df_final.withColumn("vaga",df_final.vaga[0].cast(IntegerType()))
df_final = df_final.withColumn("area_total",df_final.area_total[0].cast(IntegerType()))
df_final = df_final.withColumn("area_util",df_final.area_util[0].cast(IntegerType()))

In [13]:
df_final.printSchema()

root
 |-- andar: long (nullable = true)
 |-- area_total: integer (nullable = true)
 |-- area_util: integer (nullable = true)
 |-- banheiros: integer (nullable = true)
 |-- caracteristicas: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- endereco: struct (nullable = true)
 |    |-- bairro: string (nullable = true)
 |    |-- cep: string (nullable = true)
 |    |-- cidade: string (nullable = true)
 |    |-- estado: string (nullable = true)
 |    |-- latitude: double (nullable = true)
 |    |-- longitude: double (nullable = true)
 |    |-- pais: string (nullable = true)
 |    |-- rua: string (nullable = true)
 |    |-- zona: string (nullable = true)
 |-- id: string (nullable = true)
 |-- quartos: integer (nullable = true)
 |-- suites: integer (nullable = true)
 |-- tipo_anuncio: string (nullable = true)
 |-- tipo_unidade: string (nullable = true)
 |-- tipo_uso: string (nullable = true)
 |-- vaga: integer (nullable = true)
 |-- valores: array (nullable = true)
 

In [14]:
df_final.show(5)

+-----+----------+---------+---------+--------------------+--------------------+--------------------+-------+------+------------+------------+-----------+----+--------------------+
|andar|area_total|area_util|banheiros|     caracteristicas|            endereco|                  id|quartos|suites|tipo_anuncio|tipo_unidade|   tipo_uso|vaga|             valores|
+-----+----------+---------+---------+--------------------+--------------------+--------------------+-------+------+------------+------------+-----------+----+--------------------+
|    3|        43|       43|        1|[Academia, Churra...|{Paciência, 23585...|d2e3a3aa-09b5-45a...|      2|  NULL|       Usado| Apartamento|Residencial|   1|[{245, NULL, Vend...|
|    2|        42|       42|        1|[Churrasqueira, P...|{Paciência, 23585...|085bab2c-87ad-452...|      2|  NULL|       Usado| Apartamento|Residencial|   1|[{0, 0, Venda, 15...|
|    1|        41|       41|        1|[Portaria 24h, Co...|{Guaratiba, 23036...|18d22cbe-1b86-4

25/11/08 21:16:03 WARN TaskSetManager: Stage 5 contains a task of very large size (4526 KiB). The maximum recommended task size is 1000 KiB.


# Como as colunas "endereco" e "valores" estão compactadas, extrairemos em DataFrames diferentes para mais tarde tratamos delas

In [15]:
import pyspark.sql.functions as f

In [16]:
df_final = df_final.select('*', 'endereco.*').drop('endereco')

df_final = df_final.withColumn('valores', f.explode('valores'))
df_final = df_final.select('*', 'valores.*').drop('valores')

df_final.show(5)

+-----+----------+---------+---------+--------------------+--------------------+-------+------+------------+------------+-----------+----+---------+--------+--------------+--------------+----------+----------+----+--------------------+----------+----------+----+-----+-----+
|andar|area_total|area_util|banheiros|     caracteristicas|                  id|quartos|suites|tipo_anuncio|tipo_unidade|   tipo_uso|vaga|   bairro|     cep|        cidade|        estado|  latitude| longitude|pais|                 rua|      zona|condominio|iptu| tipo|valor|
+-----+----------+---------+---------+--------------------+--------------------+-------+------+------------+------------+-----------+----+---------+--------+--------------+--------------+----------+----------+----+--------------------+----------+----------+----+-----+-----+
|    3|        43|       43|        1|[Academia, Churra...|d2e3a3aa-09b5-45a...|      2|  NULL|       Usado| Apartamento|Residencial|   1|Paciência|23585430|Rio de Janeiro|Rio

25/11/08 21:16:03 WARN TaskSetManager: Stage 6 contains a task of very large size (4526 KiB). The maximum recommended task size is 1000 KiB.


## A equipe de ciência de dados nos solicitou que apenas as informações sobre bairro e zona da cidade fossem extraídas. Logo retiraremos as colunas não solicitadas

In [17]:
df_final = df_final.drop('cep')
df_final = df_final.drop('cidade')
df_final = df_final.drop('estado')
df_final = df_final.drop('latitude')
df_final = df_final.drop('longitude')
df_final = df_final.drop('pais')
df_final = df_final.drop('rua')

df_final.show(5)

+-----+----------+---------+---------+--------------------+--------------------+-------+------+------------+------------+-----------+----+---------+----------+----------+----+-----+-----+
|andar|area_total|area_util|banheiros|     caracteristicas|                  id|quartos|suites|tipo_anuncio|tipo_unidade|   tipo_uso|vaga|   bairro|      zona|condominio|iptu| tipo|valor|
+-----+----------+---------+---------+--------------------+--------------------+-------+------+------------+------------+-----------+----+---------+----------+----------+----+-----+-----+
|    3|        43|       43|        1|[Academia, Churra...|d2e3a3aa-09b5-45a...|      2|  NULL|       Usado| Apartamento|Residencial|   1|Paciência|Zona Oeste|       245|NULL|Venda|15000|
|    2|        42|       42|        1|[Churrasqueira, P...|085bab2c-87ad-452...|      2|  NULL|       Usado| Apartamento|Residencial|   1|Paciência|Zona Oeste|         0|   0|Venda|15000|
|    1|        41|       41|        1|[Portaria 24h, Co...|1

25/11/08 21:16:03 WARN TaskSetManager: Stage 7 contains a task of very large size (4526 KiB). The maximum recommended task size is 1000 KiB.


## A InsightPlaces permite que o(a) anunciante crie um anúncio com duas opções de valor. Assim, o(a) cliente pode criar um anúncio que mostre tanto o valor de venda do imóvel quanto o seu valor de locação, juntamente com os valores de taxa de condomínio (quando houver) e taxa de IPTU. Estes valores são diferenciados pelo campo tipo que pode assumir os valores Venda e Aluguel.

## Como se trata de um estudo sobre o preço de venda dos imóveis, o time de cientistas de dados solicitou apenas as informações do tipo VENDA. Logo faremos um filtro a partir desta coluna.

In [18]:
df_final.select('tipo').distinct().show()

25/11/08 21:16:04 WARN TaskSetManager: Stage 8 contains a task of very large size (4526 KiB). The maximum recommended task size is 1000 KiB.


+-------+
|   tipo|
+-------+
|Aluguel|
|  Venda|
+-------+



In [19]:
df_final = df_final.select('*').where(df_final['tipo'] == 'Venda')

df_final.select('tipo').distinct().show()

25/11/08 21:16:06 WARN TaskSetManager: Stage 11 contains a task of very large size (4526 KiB). The maximum recommended task size is 1000 KiB.


+-----+
| tipo|
+-----+
|Venda|
+-----+



In [20]:
df_final.show(5)

25/11/08 21:16:07 WARN TaskSetManager: Stage 14 contains a task of very large size (4526 KiB). The maximum recommended task size is 1000 KiB.


+-----+----------+---------+---------+--------------------+--------------------+-------+------+------------+------------+-----------+----+---------+----------+----------+----+-----+-----+
|andar|area_total|area_util|banheiros|     caracteristicas|                  id|quartos|suites|tipo_anuncio|tipo_unidade|   tipo_uso|vaga|   bairro|      zona|condominio|iptu| tipo|valor|
+-----+----------+---------+---------+--------------------+--------------------+-------+------+------------+------------+-----------+----+---------+----------+----------+----+-----+-----+
|    3|        43|       43|        1|[Academia, Churra...|d2e3a3aa-09b5-45a...|      2|  NULL|       Usado| Apartamento|Residencial|   1|Paciência|Zona Oeste|       245|NULL|Venda|15000|
|    2|        42|       42|        1|[Churrasqueira, P...|085bab2c-87ad-452...|      2|  NULL|       Usado| Apartamento|Residencial|   1|Paciência|Zona Oeste|         0|   0|Venda|15000|
|    1|        41|       41|        1|[Portaria 24h, Co...|1

# Por fim salvaremos o arquivo em formato parquet e csv e compararemos os dois. Como csv não suporta o modelo de estrutura da coluna CARACTERISTICAS, para este caso transformaremos em uma STRING

In [22]:
df_final.write.parquet('/home/luizh/Python/1Challenge/Data Science/Alura-Challenge-DS/Salvamentos/df_imoveis_semana1.parquet')

25/11/08 21:17:18 WARN TaskSetManager: Stage 16 contains a task of very large size (4526 KiB). The maximum recommended task size is 1000 KiB.


In [23]:
df_final = df_final.withColumn('caracteristicas', df_final['caracteristicas'].cast(StringType()))

df_final.write.csv('/home/luizh/Python/1Challenge/Data Science/Alura-Challenge-DS/Salvamentos/df_imoveis_semana1.csv')

25/11/08 21:17:21 WARN TaskSetManager: Stage 17 contains a task of very large size (4526 KiB). The maximum recommended task size is 1000 KiB.


In [24]:
%%time
df_parquet = spark.read.parquet('/home/luizh/Python/1Challenge/Data Science/Alura-Challenge-DS/Salvamentos/df_imoveis_semana1.parquet')
df_parquet.show(5)

+-----+----------+---------+---------+--------------------+--------------------+-------+------+------------+------------+-----------+----+--------------------+------------+----------+----+-----+------+
|andar|area_total|area_util|banheiros|     caracteristicas|                  id|quartos|suites|tipo_anuncio|tipo_unidade|   tipo_uso|vaga|              bairro|        zona|condominio|iptu| tipo| valor|
+-----+----------+---------+---------+--------------------+--------------------+-------+------+------------+------------+-----------+----+--------------------+------------+----------+----+-----+------+
|   10|        48|       48|        1|[Academia, Churra...|ddedd3be-c92d-4f9...|      2|     0|       Usado| Apartamento|Residencial|   0|        Santo Cristo|Zona Central|       200|NULL|Venda|324900|
|    5|        47|       47|        1|[Salão de festas,...|96d814f5-40b0-4b7...|      1|     0|       Usado| Apartamento|Residencial|   1|         Vila Isabel|  Zona Norte|        20|  20|Vend

In [25]:
%%time
df_csv = spark.read.csv('/home/luizh/Python/1Challenge/Data Science/Alura-Challenge-DS/Salvamentos/df_imoveis_semana1.csv')
df_csv.show(5)

+---+---+---+---+--------------------+--------------------+---+----+-----+-----------+-----------+----+--------------------+------------+----+----+-----+------+
|_c0|_c1|_c2|_c3|                 _c4|                 _c5|_c6| _c7|  _c8|        _c9|       _c10|_c11|                _c12|        _c13|_c14|_c15| _c16|  _c17|
+---+---+---+---+--------------------+--------------------+---+----+-----+-----------+-----------+----+--------------------+------------+----+----+-----+------+
| 10| 48| 48|  1|[Academia, Churra...|ddedd3be-c92d-4f9...|  2|   0|Usado|Apartamento|Residencial|   0|        Santo Cristo|Zona Central| 200|NULL|Venda|324900|
|  5| 47| 47|  1|[Salão de festas,...|96d814f5-40b0-4b7...|  1|   0|Usado|Apartamento|Residencial|   1|         Vila Isabel|  Zona Norte|  20|  20|Venda|325000|
| 10| 46| 46|  2|[Academia, Churra...|78f053be-6eec-4fd...|  2|   1|Usado|Apartamento|Residencial|   1|     Todos os Santos|  Zona Norte| 460|1900|Venda|329000|
|  0| 60| 60|  1|[Academia, Condom

# Como é percepitível, além de o arquivo parquet carrega mais rápido, temos que no arquivo csv não é carregado os nomes das colunas, sendo dada a função para quem for mexer nos arquivos.